# Customer Churn Perdiction Using Decision Tree, Random Forest, and Genetic Algorithm

#### Imports

In [1]:
!pip install deap -q

In [2]:
import pandas as pd                                          # Load data structures (DataFrames)
import numpy as np                                           # Support large multi-dimensional arrays

from sklearn.model_selection import train_test_split         # Split data into train/test sets
from sklearn.preprocessing import StandardScaler             # Normalize features to mean=0, var=1

from sklearn.tree import DecisionTreeClassifier             # Import decision tree classifier model
from sklearn.metrics import (                                # Open metrics import container group
    accuracy_score,                                          # Metric for overall correct predictions
    precision_score,                                         # Metric for true positive prediction rate
    recall_score,                                            # Metric for actual positive capture rate
    f1_score,                                                # Metric balancing precision and recall
    confusion_matrix,                                        # Table showing correct vs wrong predictions
    classification_report                                    # Summary text of main evaluation metrics
)                                                            # Close metrics import container group

from sklearn.ensemble import RandomForestClassifier          # Import random forest ensemble model

import random                                                # Generate pseudo-random numbers
from deap import base, creator, tools, algorithms            # Core modules for genetic algorithms

In [3]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

## Dataset Description

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.shape

(7043, 21)

In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [7]:
df['Churn'].value_counts()                                   # Identify Target Variable

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## Data Preprocessing

Target Variable: Churn 
 Yes → Customer left
 No → Customer stayed
 This confirms it is a binary classification problem.

STEP 6: Data Cleaning & Preprocessing

Handle missing values

Encode categorical variables

Feature scaling

Train–test split

In [8]:
# Reload Dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [9]:
# Remove Irrelevant Columns customerID is an identifier and does not help prediction.
df.drop(columns=['customerID'], inplace=True)

In [10]:
# Handle Missing Values
# In this dataset, TotalCharges is stored as object due to empty strings.
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [11]:
# Check missing values
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [12]:
#Drop rows with missing values
df.dropna(inplace=True)

In [13]:
# Encode Target Variable
# Convert Churn to binary format.

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [14]:
# # Encode Categorical Features (One-Hot Encoding)
# # Identify categorical columns
# categorical_cols = df.select_dtypes(include=['object']).columns

In [15]:
# Encode Categorical Features (One-Hot Encoding)                 
# Identify categorical columns                                     
categorical_cols = df.select_dtypes(include=['object', 'str']).columns 

In [16]:
# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [17]:
# Separate Features and Target
X = df.drop('Churn', axis=1)
y = df['Churn']

In [18]:
# Feature Scaling (Required for Genetic Algorithm)
# Tree-based models don’t require scaling, but Genetic Algorithms do.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [19]:
# Train–Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
# Verify Final Shapes
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (5625, 30)
Testing set shape: (1407, 30)


## Decision Tree Model

In [21]:
# Train the Decision Tree Model
# We use controlled depth to reduce overfitting.

dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=6,
    random_state=42
)

dt_model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [22]:
# 7.3 Make Predictions
y_pred_dt = dt_model.predict(X_test)

In [23]:
# Model Evaluation
# Accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_accuracy

0.7853589196872779

In [24]:
# Precision, Recall, and F1-Score
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

dt_precision, dt_recall, dt_f1
print("Desision Tree Precision :", dt_precision)
print("Desision Tree Recall :", dt_recall)
print("Desision Tree F1-Score :", dt_f1)

Desision Tree Precision : 0.6111111111111112
Desision Tree Recall : 0.5294117647058824
Desision Tree F1-Score : 0.5673352435530086


In [25]:
#Confusion Matrix
conf_matrix_dt = confusion_matrix(y_test, y_pred_dt)
conf_matrix_dt

array([[907, 126],
       [176, 198]])

In [26]:
report_dict = classification_report(y_test, y_pred_dt, output_dict=True)

# Convert dictionary to DataFrame
report_dt = pd.DataFrame(report_dict).transpose()
report_dt = report_dt.round(4)
report_dt

,precision,recall,f1-score,support
0,0.8375,0.8780,0.8573,1033.0000
1,0.6111,0.5294,0.5673,374.0000
accuracy,0.7854,0.7854,0.7854,0.7854
macro avg,0.7243,0.7037,0.7123,1407.0000
weighted avg,0.7773,0.7854,0.7802,1407.0000


## Random Forest Model

In [27]:
rf = RandomForestClassifier(                                 # Initialize the Random Forest model object
    n_estimators=100,                                        # Set the number of decision trees to 100
    random_state=42                                          # Set a random seed to guarantee reproducibility
)                                                            # Close the model initialization configuration
rf.fit(X_train, y_train)                                     # Train the model using the training datasets

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [28]:
y_pred_rf = rf.predict(X_test)                               # Generate predictions using the test feature dataset

In [29]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)              # Calculate the overall prediction accuracy score
rf_accuracy

0.7896233120113717

In [30]:
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)         # Compute the confusion matrix evaluation table
conf_matrix_rf

array([[917, 116],
       [180, 194]])

In [31]:
# Precision, Recall, and F1-Score
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

dt_precision, dt_recall, dt_f1
print("Random Forest Precision :", rf_precision)
print("Random Forest Recall :", rf_recall)
print("Random Forest F1-Score :", rf_f1)

Random Forest Precision : 0.6258064516129033
Random Forest Recall : 0.5187165775401069
Random Forest F1-Score : 0.5672514619883041


In [32]:
report_dict = classification_report(y_test, y_pred_rf, output_dict=True)

# Convert dictionary to DataFrame
report_rf = pd.DataFrame(report_dict).transpose()
report_rf = report_rf.round(4) 
report_rf

,precision,recall,f1-score,support
0,0.8359,0.8877,0.8610,1033.0000
1,0.6258,0.5187,0.5673,374.0000
accuracy,0.7896,0.7896,0.7896,0.7896
macro avg,0.7309,0.7032,0.7141,1407.0000
weighted avg,0.7801,0.7896,0.7829,1407.0000


## Genetic Algorithm Model

 #### GA is used for optimization, not classification

In [33]:
# 2. Define GA Fitness & Individual
# Avoid re-creation error in Jupyter
if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMax)

In [34]:
# 3. GA Toolbox Setup
toolbox = base.Toolbox()

n_features = X_train.shape[1]

# Binary chromosome (0/1)
toolbox.register("attr_bool", random.randint, 0, 1)

toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=n_features
)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [35]:
# 4. Fitness Function (Accuracy)
def evaluate(individual):
    selected_features = [i for i, bit in enumerate(individual) if bit == 1]

    # Penalize empty feature set
    if len(selected_features) == 0:
        return 0.0,

    X_train_sel = X_train[:, selected_features]
    X_test_sel = X_test[:, selected_features]

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_sel, y_train)

    y_pred = model.predict(X_test_sel)
    acc = accuracy_score(y_test, y_pred)

    return acc,

In [36]:
# 5. Register GA Operators
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [37]:
# 6. Run Genetic Algorithm
population = toolbox.population(n=30)
generations = 20

algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.5,
    mutpb=0.2,
    ngen=generations,
    verbose=True
)

gen	nevals
0  	30    
1  	17    
2  	14    
3  	18    
4  	17    
5  	22    
6  	17    
7  	22    
8  	24    
9  	19    
10 	19    
11 	21    
12 	22    
13 	14    
14 	18    
15 	17    
16 	19    
17 	19    
18 	12    
19 	22    
20 	18    


([[0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   1,
   0,
   1,
   1,
   0,
   0,
   0,
   1],
  [0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   1,
   0,
   1,
   1,
   1,
   0,
   0,
   1],
  [0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   1,
   0,
   1,
   1,
   1,
   0,
   0,
   1],
  [0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   1,
   0,
   1,
   1,
   1,
   0,
   0,
   1],
  [0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   1,
   0,
   1,
   1,
   1,
   0,
   0,
   1],
  [0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   1,
   1,
   1,
   0,
   1,
   0,

In [38]:
# 7. Extract Best Feature Set
best_individual = tools.selBest(population, 1)[0]
selected_features = [i for i, bit in enumerate(best_individual) if bit == 1]
print("Selected features:", len(selected_features))

Selected features: 14


In [39]:
# 8. Final GA-Based Model Evaluation
X_train_ga = X_train[:, selected_features]
X_test_ga = X_test[:, selected_features]

ga_model = DecisionTreeClassifier(random_state=42)
ga_model.fit(X_train_ga, y_train)

y_pred_ga = ga_model.predict(X_test_ga)

ga_accuracy= accuracy_score(y_test, y_pred_ga)
print("Genetic Algorithm Accuracy:", ga_accuracy)

Genetic Algorithm Accuracy: 0.7789623312011372


In [40]:
ga_accuracy = accuracy_score(y_test, y_pred_ga)
print(ga_accuracy)

0.7789623312011372


In [41]:
conf_matrix_ga = confusion_matrix(y_test, y_pred_ga)
conf_matrix_ga

array([[895, 138],
       [173, 201]])

In [42]:
ga_precision = precision_score(y_test, y_pred_ga)
ga_recall = recall_score(y_test, y_pred_ga)
ga_f1 = f1_score(y_test, y_pred_ga)

dt_precision, dt_recall, dt_f1
print("Genetic Algorithm Precision :", ga_precision)
print("Genetic Algorithm Recall :", ga_recall)
print("Genetic Algorithm F1-Score :", ga_f1)

Genetic Algorithm Precision : 0.5929203539823009
Genetic Algorithm Recall : 0.5374331550802139
Genetic Algorithm F1-Score : 0.5638148667601683


In [43]:
report_dict = classification_report(y_test, y_pred_ga, output_dict=True)

# Convert dictionary to DataFrame
report_ga = pd.DataFrame(report_dict).transpose()
report_ga = report_ga.round(4)
report_ga

,precision,recall,f1-score,support
0,0.8380,0.8664,0.8520,1033.000
1,0.5929,0.5374,0.5638,374.000
accuracy,0.7790,0.7790,0.7790,0.779
macro avg,0.7155,0.7019,0.7079,1407.000
weighted avg,0.7729,0.7790,0.7754,1407.000


## Model Comparison

In [44]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Genetic Algorithm"],
    "Accuracy": [dt_accuracy, rf_accuracy, ga_accuracy]
})

results

,Model,Accuracy
0,Decision Tree,0.785359
1,Random Forest,0.789623
2,Genetic Algorithm,0.778962
